# Hamilton DAG — Feature Engineering

This notebook explores the **Hamilton-based feature engineering layer**:

- Building a `FeaturePipeline` from the Hamilton DAG
- Visualizing the DAG lineage graph
- Computing all ~35 features for a single ticker
- Visualizing technical indicators (RSI, MACD, Bollinger Bands)
- Feature correlation matrix

**Prerequisites:** Ingested price data in `data/lake/` (run `uv run equity ingest`).

In [1]:
import sys
sys.path.insert(0, "../src")

from equity_lake.features.pipeline import FeaturePipeline
from equity_lake.features.dag import clean_02, features_03, raw_01
from equity_lake.features.indicators import rsi, macd, bollinger_bands, atr, ema
from equity_lake.storage.duckdb import EquityDataDB
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

print("Feature modules loaded successfully")

Feature modules loaded successfully


## 1. Explore the Hamilton DAG

The layered DAG modules (`raw_01`, `clean_02`, `features_03`) define the feature graph. Each function's parameter names declare its dependencies.

In [2]:
# List all Hamilton nodes
import inspect

nodes = []
for module in (raw_01, clean_02, features_03):
    for name, func in inspect.getmembers(module, inspect.isfunction):
        if not name.startswith("_"):
            sig = inspect.signature(func)
            deps = list(sig.parameters.keys())
            doc = (func.__doc__ or "").strip().split("\n")[0][:80]
            nodes.append({"Node": name, "Dependencies": deps, "Description": doc})

nodes_df = pd.DataFrame(nodes)
print(f"Total Hamilton nodes: {len(nodes_df)}")
nodes_df

Total Hamilton nodes: 48


,Node,Dependencies,Description
0,atr,"[high, low, close, length]",
1,atr_14,"[high, low, close]",
2,bb_lower,[bollinger_frame],
3,bb_middle,[bollinger_frame],
4,bb_pct,"[close, bb_upper, bb_lower]",
5,bb_upper,[bollinger_frame],
6,bb_width,"[bb_upper, bb_lower, bb_middle]",
7,bollinger_bands,"[series, length, std]",
8,bollinger_frame,[close],
9,close,[price_data],


## 2. DAG Lineage Visualization

Export the Hamilton DAG as an image showing all node dependencies.

In [3]:
pipeline = FeaturePipeline()

# Export lineage to PNG
try:
    lineage_path = pipeline.export_lineage("hamilton_dag.png")
    print(f"DAG lineage exported to: {lineage_path}")
    from IPython.display import Image, display
    display(Image(filename=lineage_path))
except Exception as e:
    print(f"Lineage export: {e}")
    print("(graphviz may not be installed — pip install graphviz)")

 graphviz is required for visualizing the function graph. Install it with:

  pip install "sf-hamilton[visualization]" or pip install graphviz 

Traceback (most recent call last):
  File "/Users/minghao/Desktop/personal/equity_lake/.venv/lib/python3.12/site-packages/hamilton/graph.py", line 921, in display
    import graphviz  # noqa: F401
    ^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'graphviz'


DAG lineage exported to: hamilton_dag.png
Lineage export: [Errno 2] No such file or directory: 'hamilton_dag.png'
(graphviz may not be installed — pip install graphviz)


## 3. Compute Features for a Single Ticker

Load price data and compute all features through the Hamilton DAG.

In [4]:
# Load sample price data
ticker = "AAPL"

with EquityDataDB() as db:
    try:
        price_data = db.query(
            "SELECT date, open, high, low, close, volume "
            "FROM us_equity "
            "WHERE ticker = '" + ticker + "' "
            "ORDER BY date"
        )
        print(f"Loaded {len(price_data)} rows for {ticker}")
        print(f"Date range: {price_data['date'].min()} to {price_data['date'].max()}")
    except Exception as e:
        print(f"Query error: {e}")
        print("Run ingestion first: dotenvx run -- uv run equity ingest")
        price_data = pd.DataFrame()

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Loaded 2518 rows for AAPL
Date range: 2016-06-06 00:00:00 to 2026-06-04 00:00:00


In [5]:
if not price_data.empty:
    features_to_compute = [
        "rsi_14", "macd", "macd_signal", "macd_histogram",
        "bb_upper", "bb_middle", "bb_lower", "bb_width", "bb_pct",
        "atr_14", "roc_5", "roc_10", "roc_20",
        "return_1d", "return_5d", "return_10d", "return_20d",
        "volatility_20", "volume_ratio", "obv",
        "overnight_return", "intraday_return",
        "day_of_week", "month", "quarter",
    ]

    result = pipeline.compute(
        price_data=price_data,
        features=features_to_compute,
    )

    print(f"Computed {len(result.columns)} features over {len(result)} dates")
    print(f"\nFeature columns: {list(result.columns)}")
    result.tail(5)
else:
    print("No price data available — skipping feature computation")

[rsi_14:range_validator] validator failed. Message was: Series contains 2504 values in range (0,100), and 14 outside.. Diagnostic information is: {'range': (0, 100), 'in_range': 2504, 'out_range': 14, 'data_size': 2518}.


/Users/minghao/Desktop/personal/equity_lake/docs/notebooks/../../src/equity_lake/features/indicators.py:63: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return series.pct_change(length) * 100
/Users/minghao/Desktop/personal/equity_lake/docs/notebooks/../../src/equity_lake/features/hamilton_features.py:120: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return close.pct_change(1)
/Users/minghao/Desktop/personal/equity_lake/docs/notebooks/../../src/equity_lake/features/hamilton_features.py:124: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a

Computed 26 features over 2518 dates

Feature columns: ['rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 'bb_pct', 'atr_14', 'roc_5', 'roc_10', 'roc_20', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'volatility_20', 'volume_ratio', 'obv', 'overnight_return', 'intraday_return', 'day_of_week', 'month', 'quarter', 'feature_schema_version']


## 4. RSI (Relative Strength Index)

Visualize RSI_14 with overbought/oversold zones.

In [6]:
if not price_data.empty and "rsi_14" in result.columns:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.7, 0.3], vertical_spacing=0.05)

    # Price
    fig.add_trace(go.Candlestick(
        x=price_data["date"], open=price_data["open"], high=price_data["high"],
        low=price_data["low"], close=price_data["close"], name="OHLC",
    ), row=1, col=1)

    # RSI
    fig.add_trace(go.Scatter(
        x=result.index if "date" not in result.columns else result["date"],
        y=result["rsi_14"], name="RSI(14)", line=dict(color="purple", width=1),
    ), row=2, col=1)

    # RSI zones
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1, annotation_text="Overbought")
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1, annotation_text="Oversold")

    fig.update_layout(title=f"{ticker} — Price & RSI(14)", height=600, xaxis_rangeslider_visible=False)
    fig.show()
else:
    print("RSI data not available")

## 5. MACD

MACD line, signal line, and histogram.

In [7]:
if not price_data.empty and "macd" in result.columns:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.7, 0.3], vertical_spacing=0.05)

    fig.add_trace(go.Scatter(
        x=price_data["date"], y=price_data["close"],
        name="Close", line=dict(width=1),
    ), row=1, col=1)

    dates = result.index if "date" not in result.columns else result["date"]
    fig.add_trace(go.Scatter(x=dates, y=result["macd"], name="MACD", line=dict(color="blue")), row=2, col=1)
    fig.add_trace(go.Scatter(x=dates, y=result["macd_signal"], name="Signal", line=dict(color="orange")), row=2, col=1)

    colors = ["green" if v >= 0 else "red" for v in result["macd_histogram"].fillna(0)]
    fig.add_trace(go.Bar(x=dates, y=result["macd_histogram"], name="Histogram", marker_color=colors), row=2, col=1)

    fig.update_layout(title=f"{ticker} — Price & MACD", height=600)
    fig.show()
else:
    print("MACD data not available")

## 6. Bollinger Bands

Price with upper/middle/lower Bollinger Bands.

In [8]:
if not price_data.empty and "bb_upper" in result.columns:
    dates = price_data["date"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dates, y=price_data["close"], name="Close", line=dict(width=1.5)))

    fig.add_trace(go.Scatter(x=dates, y=result["bb_upper"], name="BB Upper",
                             line=dict(dash="dash", color="gray", width=1)))
    fig.add_trace(go.Scatter(x=dates, y=result["bb_middle"], name="BB Middle",
                             line=dict(dash="dash", color="blue", width=1)))
    fig.add_trace(go.Scatter(x=dates, y=result["bb_lower"], name="BB Lower",
                             line=dict(dash="dash", color="gray", width=1)))

    fig.add_trace(go.Scatter(
        x=pd.concat([dates, dates.iloc[::-1]]),
        y=pd.concat([result["bb_upper"], result["bb_lower"].iloc[::-1]]),
        fill="toself", fillcolor="rgba(128,128,128,0.1)",
        line=dict(color="rgba(255,255,255,0)"), showlegend=False,
    ))

    fig.update_layout(title=f"{ticker} — Bollinger Bands (20, 2σ)", height=600, yaxis_title="Price")
    fig.show()
else:
    print("Bollinger Bands data not available")

## 7. Feature Correlation Matrix

How correlated are the generated features?

In [9]:
if not result.empty:
    numeric_cols = result.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) > 2:
        corr = result[numeric_cols].corr()

        fig = px.imshow(
            corr,
            title="Feature Correlation Matrix",
            color_continuous_scale="RdBu_r",
            zmin=-1, zmax=1,
            aspect="auto",
        )
        fig.update_layout(height=700, width=900)
        fig.show()
    else:
        print("Not enough numeric features for correlation")
else:
    print("No feature data computed")

## 8. Feature Heatmap Over Time

Visualize how features evolve over the last 60 days.

In [10]:
if not result.empty:
    numeric_cols = result.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) >= 3:
        recent = result[numeric_cols].tail(60).T
        recent = (recent - recent.min(axis=1).values.reshape(-1, 1)) / (recent.max(axis=1).values.reshape(-1, 1) - recent.min(axis=1).values.reshape(-1, 1) + 1e-10)

        fig = px.imshow(
            recent.values,
            x=list(range(recent.shape[1])),
            y=numeric_cols,
            title="Normalized Features Over Last 60 Days",
            color_continuous_scale="Viridis",
            aspect="auto",
        )
        fig.update_layout(height=max(600, len(numeric_cols) * 15), yaxis_title="Feature", xaxis_title="Day")
        fig.show()
else:
    print("No feature data for heatmap")

## Next Steps

- **[05-feature-engineering-deep-dive.ipynb](05-feature-engineering-deep-dive.ipynb)** — Sentiment features & cross-modal analysis
- **[06-ml-prediction.ipynb](06-ml-prediction.ipynb)** — Use features for ML predictions
- **[07-signal-scanning.ipynb](07-signal-scanning.ipynb)** — Generate trading signals from features